In [28]:
"""Baixa os discursos da Câmara dos Deputados API Dados Abertos)."""
from concurrent.futures import ThreadPoolExecutor
from datetime import date, timedelta

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE = "https://dadosabertos.camara.leg.br/api/v2"

hoje = date.today()
params_base = {
    "dataInicio": '2026-10-10',
    "dataFim": '2026-11-11',
    "itens": 100,
    "ordem": "ASC",
    "ordenarPor": "dataHoraInicio",
}

# Sessão com retentativas (a API às vezes responde 429/5xx)
session = requests.Session()
session.headers["Accept"] = "application/json"
session.mount(
    "https://",
    HTTPAdapter(max_retries=Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])),
)


def baixar_paginado(url, params):
    """Percorre todas as páginas seguindo o link 'next' devolvido pela API."""
    registros = []
    while url:
        r = session.get(url, params=params, timeout=60)
        r.raise_for_status()
        corpo = r.json()
        registros.extend(corpo["dados"])
        url = next((l["href"] for l in corpo.get("links", []) if l["rel"] == "next"), None)
        params = None  # o link 'next' já carrega os parâmetros
    return registros


def discursos_geral():
    return baixar_paginado(f"{BASE}/discursos", params_base)


def discursos_por_deputado():
    """Plano B: consulta /deputados/{id}/discursos para cada deputado em exercício."""
    deputados = baixar_paginado(f"{BASE}/deputados", {"itens": 100})

    def um(dep):
        regs = baixar_paginado(f"{BASE}/deputados/{dep['id']}/discursos", params_base)
        for reg in regs:
            reg["idDeputado"], reg["nomeDeputado"] = dep["id"], dep["nome"]
            reg["siglaPartido"], reg["siglaUf"] = dep["siglaPartido"], dep["siglaUf"]
        return regs

    with ThreadPoolExecutor(max_workers=8) as ex:
        return [reg for lista in ex.map(um, deputados) for reg in lista]


try:
    dados = discursos_geral()
except requests.HTTPError as e:
    print(f"/discursos falhou ({e}); usando /deputados/{{id}}/discursos...")
    dados = discursos_por_deputado()

# json_normalize achata campos aninhados (ex.: tipoDiscurso, faseEvento)
df = pd.json_normalize(dados)
print(f"Período: {params_base['dataInicio']} a {params_base['dataFim']}")
print(df.shape)
print(df.head())

df.to_csv(f"discursos_camara_{params_base['dataInicio']}_{params_base['dataFim']}.csv", index=False)

/discursos falhou (405 Client Error: Method Not Allowed for url: https://dadosabertos.camara.leg.br/api/v2/discursos?dataInicio=2026-10-10&dataFim=2026-11-11&itens=100&ordem=ASC&ordenarPor=dataHoraInicio); usando /deputados/{id}/discursos...
Período: 2026-10-10 a 2026-11-11
(961, 17)
     dataHoraInicio dataHoraFim uriEvento               tipoDiscurso  \
0  2026-11-04T15:04        None                  BREVES COMUNICAÇÕES   
1  2026-11-10T17:56        None                           PELA ORDEM   
2  2026-10-14T17:32        None                            DISCUSSÃO   
3  2026-10-14T19:04        None            ENCAMINHAMENTO DE VOTAÇÃO   
4  2026-10-14T19:04        None                           PELA ORDEM   

                                            urlTexto urlAudio urlVideo  \
0  https://imagem.camara.gov.br/dc_20b.asp?largur...     None     None   
1  https://imagem.camara.gov.br/dc_20b.asp?largur...     None     None   
2  https://imagem.camara.gov.br/dc_20b.asp?largur...     Non

In [ ]:
df.shape

(243, 17)

In [ ]:
df.iloc[0]

,0
dataHoraInicio,2026-11-10T17:56
dataHoraFim,None
uriEvento,
tipoDiscurso,PELA ORDEM
urlTexto,https://imagem.camara.gov.br/dc_20b.asp?largur...
urlAudio,None
urlVideo,None
keywords,"Deputado federal,Atuação parlamentar,Candidato..."
sumario,O Deputado parabenizou Vicentinho Júnior pela ...
transcricao,O SR. ADOLFO VIANA (Bloco/PSDB - BA. Pela orde...
